# Explainable Industrial Defect Inspector — Colab Setup

Mounts Drive, clones the repo, installs dependencies, and sets up the project.

- **Data:** `defect_inspector/data/mvtec` (persisted in Drive)
- **Results:** `defect_inspector/outputs/` (persisted in Drive)
- **Code:** cloned from GitHub to `/content/xAI_Defects`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/defect_inspector"
LOCAL_DIR  = "/content/xAI_Defects"

DRIVE_DATA    = os.path.join(DRIVE_ROOT, "data")
DRIVE_OUTPUTS = os.path.join(DRIVE_ROOT, "outputs")

os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")
print(f"Exists: {os.path.exists(DRIVE_ROOT)}")

## Clone / Pull Repository

In [ ]:
REPO_URL = "https://github.com/AKIF-jk/xAI_Defects.git"

%cd /content
if os.path.exists(LOCAL_DIR):
    %cd {LOCAL_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {LOCAL_DIR}

## Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet
!pip install open_clip_torch timm captum shap albumentations fastapi uvicorn python-multipart gradio faiss-cpu anthropic matplotlib seaborn scikit-learn --quiet

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:      {torch.cuda.get_device_name(0)}")
    print(f"GPU count:     {torch.cuda.device_count()}")

In [ ]:
import open_clip, captum, shap, gradio
print(f"torch:              {torch.__version__}")
print(f"open_clip_torch:    {open_clip.__version__}")
print(f"captum:             {captum.__version__}")
print(f"shap:               {shap.__version__}")
print(f"gradio:             {gradio.__version__}")

## Create Project Folders

In [ ]:
folders = [
    "data/mvtec",
    "src/data", "src/model", "src/xai", "src/api",
    "outputs/heatmaps", "outputs/shap", "outputs/results",
]

for f in folders:
    os.makedirs(os.path.join(DRIVE_ROOT, f), exist_ok=True)

print("Drive folder structure created:")
for f in folders:
    print(f"  {os.path.join(DRIVE_ROOT, f)}")

In [ ]:
import sys

SRC_PATH = os.path.join(DRIVE_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

LOCAL_SRC = f"{LOCAL_DIR}/src"
if os.path.exists(LOCAL_SRC) and LOCAL_SRC not in sys.path:
    sys.path.insert(0, LOCAL_SRC)

print(f"sys.path entries:\n  {SRC_PATH}\n  {LOCAL_SRC}")

## Download MVTec AD Dataset

In [ ]:
!pip install kagglehub -q

MVTEC_DIR = f"{DRIVE_ROOT}/data/mvtec"
EXTRACTED = os.path.join(MVTEC_DIR, "mvtec_anomaly_detection")

if os.path.exists(EXTRACTED):
    print(f"MVTec AD already in Drive: {EXTRACTED}")
else:
    os.makedirs(MVTEC_DIR, exist_ok=True)
    print("Downloading MVTec AD from Kaggle...")
    cache_path = kagglehub.dataset_download("ipythonx/mvtec-ad")
    !cp -r "{cache_path}"/* "{EXTRACTED}"
    print(f"MVTec AD ready at: {EXTRACTED}")

In [ ]:
import glob

CATEGORIES = ["bottle","cable","capsule","carpet","grid","hazelnut","leather",
              "metal_nut","pill","screw","tile","toothbrush","transistor","wood","zipper"]

found = [c for c in CATEGORIES if os.path.isdir(os.path.join(EXTRACTED, c))]
print(f"Categories found: {len(found)}/15")

category_stats = {}
for cat in CATEGORIES:
    cat_dir = os.path.join(EXTRACTED, cat)
    if not os.path.isdir(cat_dir):
        continue
    train_dir = os.path.join(cat_dir, "train", "good")
    train_count = len(glob.glob(os.path.join(train_dir, "*.png")))
    test_dir = os.path.join(cat_dir, "test")
    defect_dirs = sorted(d for d in os.listdir(test_dir)
                         if os.path.isdir(os.path.join(test_dir, d)) and d != "good")
    test_good = len(glob.glob(os.path.join(test_dir, "good", "*.png")))
    test_anom = sum(len(glob.glob(os.path.join(test_dir, d, "*.png"))) for d in defect_dirs)
    category_stats[cat] = {"train": train_count, "normal_test": test_good,
                          "anom_test": test_anom, "defects": defect_dirs}

print(f"{'Category':<15} {'Train':>6} {'Test':>6}")
print("-" * 29)
for cat, stats in category_stats.items():
    total_test = stats["normal_test"] + stats["anom_test"]
    print(f"{cat:<15} {stats['train']:>6} {total_test:>6}")

## Validate Modules

In [ ]:
sys.path.insert(0, f'{LOCAL_DIR}/src/data')
from mvtec_dataset import MVTecDataset

train_ds = MVTecDataset(EXTRACTED, "bottle", split="train")
test_ds  = MVTecDataset(EXTRACTED, "bottle", split="test")

print(f"Bottle - Train: {len(train_ds)}, Test: {len(test_ds)}")

In [ ]:
%cd {LOCAL_DIR}/src/model

## Run Backbone & Evaluation

In [ ]:
!python3 {LOCAL_DIR}/src/model/backbone.py --data_dir {EXTRACTED}

In [ ]:
!python3 {LOCAL_DIR}/src/model/adaptclip.py

In [ ]:
!python3 {LOCAL_DIR}/src/eval/zero_shot.py --data_dir {EXTRACTED}

In [ ]:
!python3 {LOCAL_DIR}/src/eval/prompt_tuning.py --data_dir {EXTRACTED}

In [ ]:
!python3 {LOCAL_DIR}/src/eval/test_memory_bank.py --data_dir {EXTRACTED}

In [ ]:
!python3 {LOCAL_DIR}/src/eval/test_score_maps.py --data_dir {EXTRACTED} --category pill --n_shots 32 --patch_layer -2

## Run XAI (Grad-CAM & SHAP)

In [ ]:
!python3 {LOCAL_DIR}/src/xai/gradcam.py --data_dir {EXTRACTED}

In [ ]:
!python3 {LOCAL_DIR}/src/xai/shap_explainer.py --data_dir {EXTRACTED} --grid_size 13 --n_evals 200

## Generate XAI Gallery

In [ ]:
!python3 {LOCAL_DIR}/generate_xai_gallery.py --data_dir {EXTRACTED} --categories carpet,grid,transistor,wood

In [ ]:
!python3 {LOCAL_DIR}/scripts/export_demo_bundle.py --data_dir {EXTRACTED} --categories carpet,grid,transistor,wood